In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [16]:
# !ls kss/wavs

In [8]:
!wc -l kss/metadata.csv

12854 kss/metadata.csv


In [14]:
rows = pd.read_csv('kss/metadata.csv', sep = '|', header = None).to_dict(orient = 'records')
rows[0]

{0: '1_0000', 1: '그는 괜찮은 척하려고 애쓰는 것 같았다.', 2: '그는 괜찮은 척하려고 애쓰는 것 같았다.'}

In [13]:
!mkdir kss_audio

In [21]:
def loop(rows):
    rows, _ = rows
    data = []
    for r in tqdm(rows):
        try:
            f = f"kss/wavs/{r[0]}.wav"
    
            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
    
            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('kss_audio', audio_filename)
            
            t = r[1].strip()
            if len(t) < 2:
                continue
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': "KSS"
            })
            
        except Exception as e:
            print(e)
            pass
            
    return data
            

In [22]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 23.72it/s]


In [24]:
data = multiprocessing(rows, loop, cores = 20)

100%|██████████| 642/642 [00:40<00:00, 15.90it/s]


In [25]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'kss_audio/kss_wavs_1_0000.mp3',
 'text': '그는 괜찮은 척하려고 애쓰는 것 같았다.',
 'speaker': 'KSS'}

In [26]:
dataset.push_to_hub('malaysia-ai/Korean-Single-Speaker-TTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 200.75ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  495kB /  495kB, 1.59MB/s  
Processing Files (1 / 1): 100%|██████████|  495kB /  495kB, 1.24MB/s  
New Data Upload: 100%|██████████|  495kB /  495kB, 1.24MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Korean-Single-Speaker-TTS/commit/e0843762f56393f72ee6e8896035db7e94633934', commit_message='Upload dataset', commit_description='', oid='e0843762f56393f72ee6e8896035db7e94633934', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Korean-Single-Speaker-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Korean-Single-Speaker-TTS'), pr_revision=None, pr_num=None)

In [30]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'KSS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 203.58ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  495kB /  495kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.24s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/514c45093dbadce422a2286f0266d38df837e78d', commit_message='Upload dataset', commit_description='', oid='514c45093dbadce422a2286f0266d38df837e78d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [28]:
audio_files = [d['audio_filename'] for d in data]

with open('KSS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [32]:
# !zip -rq kss_audio.zip kss_audio

In [31]:
# !hf upload malaysia-ai/Korean-Single-Speaker-TTS kss_audio.zip --repo-type=dataset

In [34]:
# !zip -rq kss_audio_neucodec.zip kss_audio_neucodec

In [36]:
# !hf upload malaysia-ai/Multilingual-TTS kss_audio_neucodec.zip --repo-type=dataset